In [28]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
!pip install timm -q


In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
import h5py
import numpy as np
from PIL import Image
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [31]:
# HDF5 dataset paths
TRAIN_H5 = "/content/drive/MyDrive/morph_project/processed/train.h5"
TEST_H5  = "/content/drive/MyDrive/morph_project/processed/test.h5"

# Model checkpoints (based on your directory listing and earlier info)
SWIN_CKPT       = "/content/drive/MyDrive/morph_project/swin224_recall_tuned.pth"
CONVNEXTV2_CKPT = "/content/drive/MyDrive/morph_project/models/convnextv2_best_224_finetuned.pth"
VIT_CKPT        = "/content/drive/MyDrive/morph_project/models/vit_advanced.pth"
EFFNET_CKPT     = "/content/drive/MyDrive/morph_project/models/efficientnet_b3_tuned_baseline.pth"

# Where to save the fusion head
FUSION_SAVE_PATH = "/content/drive/MyDrive/morph_project/models/fusion_4models_logits_head.pth"


In [32]:
img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class HDF5Dataset(Dataset):
    def __init__(self, h5_path, transform=None):
        self.h5_path = h5_path
        self.transform = transform

        with h5py.File(self.h5_path, "r") as f:
            self.length = f["X"].shape[0]

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        with h5py.File(self.h5_path, "r") as f:
            img = f["X"][idx]      # (224, 224, 3)
            label = int(f["y"][idx])

        # ensure uint8 for PIL
        if img.dtype != np.uint8:
            img = img.astype(np.uint8)

        img = Image.fromarray(img)

        if self.transform is not None:
            img = self.transform(img)

        return img, label


In [33]:
batch_size = 32

train_dataset = HDF5Dataset(TRAIN_H5, transform=img_transform)
test_dataset  = HDF5Dataset(TEST_H5, transform=img_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))


Train size: 7177
Test size: 1803


In [34]:
class Swin224Classifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.backbone = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=False,
            num_classes=0   # feature extractor
        )
        in_features = self.backbone.num_features

        self.classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        feats = self.backbone(x)
        logits = self.classifier(feats)
        return logits


In [35]:
class ConvNeXtV2Classifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.backbone = timm.create_model(
            "convnextv2_tiny.fcmae_ft_in1k",
            pretrained=False,
            num_classes=0
        )
        in_features = self.backbone.num_features

        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        feats = self.backbone(x)
        logits = self.classifier(feats)
        return logits


In [36]:
def create_vit_model(num_classes=2):
    model = timm.create_model(
        "vit_base_patch16_224",
        pretrained=False,
        num_classes=num_classes
    )
    return model


In [37]:
def create_efficientnet_model(num_classes=2):
    model = timm.create_model(
        "efficientnet_b3",
        pretrained=False,
        num_classes=num_classes
    )
    return model


In [38]:
# Swin
swin = Swin224Classifier(num_classes=2).to(device)
swin_state = torch.load(SWIN_CKPT, map_location=device)
swin.load_state_dict(swin_state)
swin.eval()

# ConvNeXtV2
convnext = ConvNeXtV2Classifier(num_classes=2).to(device)
conv_state = torch.load(CONVNEXTV2_CKPT, map_location=device)
convnext.load_state_dict(conv_state)
convnext.eval()

# ViT (timm, pure model)
vit = create_vit_model(num_classes=2).to(device)
vit_state = torch.load(VIT_CKPT, map_location=device)
vit.load_state_dict(vit_state)
vit.eval()

# EfficientNet-B3 (timm, pure model)
effnet = create_efficientnet_model(num_classes=2).to(device)
eff_state = torch.load(EFFNET_CKPT, map_location=device)
effnet.load_state_dict(eff_state)
effnet.eval()

# Freeze all 4 models
for model in [swin, convnext, vit, effnet]:
    for p in model.parameters():
        p.requires_grad = False

print("✅ All 4 models loaded and frozen.")


✅ All 4 models loaded and frozen.


In [39]:
class LogitFusionHead(nn.Module):
    def __init__(self, num_models=4, num_classes=2):
        super().__init__()
        in_features = num_models * num_classes  # 4 * 2 = 8

        self.fusion = nn.Sequential(
            nn.Linear(in_features, 16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(16, num_classes)
        )

    def forward(self, logits_list):
        # logits_list: [logits_swin, logits_conv, logits_vit, logits_eff]
        x = torch.cat(logits_list, dim=1)  # [B, 8]
        out = self.fusion(x)
        return out

fusion_head = LogitFusionHead(num_models=4, num_classes=2).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(fusion_head.parameters(), lr=1e-3, weight_decay=1e-4)


In [40]:
def train_fusion(num_epochs=10):
    fusion_head.train()

    for epoch in range(1, num_epochs + 1):
        running_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs}")
        for imgs, labels in pbar:
            imgs = imgs.to(device)
            labels = labels.to(device)

            # Get logits from all four frozen models
            with torch.no_grad():
                logits_swin = swin(imgs)        # [B, 2]
                logits_conv = convnext(imgs)    # [B, 2]
                logits_vit  = vit(imgs)         # [B, 2]
                logits_eff  = effnet(imgs)      # [B, 2]

            fusion_logits = fusion_head([logits_swin, logits_conv, logits_vit, logits_eff])

            loss = criterion(fusion_logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)
            preds = fusion_logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        epoch_loss = running_loss / total
        epoch_acc = correct / total
        print(f"Epoch {epoch}: Loss={epoch_loss:.4f} | Acc={epoch_acc:.4f}")


In [41]:
train_fusion(num_epochs=10)


Epoch 1/10: 100%|██████████| 225/225 [08:04<00:00,  2.15s/it, loss=0.0534]


Epoch 1: Loss=1.1039 | Acc=0.8459


Epoch 2/10: 100%|██████████| 225/225 [06:45<00:00,  1.80s/it, loss=0.3572]


Epoch 2: Loss=0.8493 | Acc=0.8984


Epoch 3/10: 100%|██████████| 225/225 [06:42<00:00,  1.79s/it, loss=0.1555]


...

Epoch 8: Loss=0.3618 | Acc=0.9288


Epoch 9/10: 100%|██████████| 225/225 [06:32<00:00,  1.75s/it, loss=0.1116]


Epoch 9: Loss=0.3499 | Acc=0.9275


Epoch 10/10: 100%|██████████| 225/225 [06:36<00:00,  1.76s/it, loss=0.0720]

Epoch 10: Loss=0.3213 | Acc=0.9328


In [45]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score
)

def evaluate_fusion(loader):
    fusion_head.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Evaluating Fusion Model"):
            imgs = imgs.to(device)
            labels = labels.to(device)

            # collect logits from all 4 models
            logits_swin = swin(imgs)
            logits_conv = convnext(imgs)
            logits_vit  = vit(imgs)
            logits_eff  = effnet(imgs)

            # fusion logits
            fusion_logits = fusion_head([
                logits_swin, logits_conv, logits_vit, logits_eff
            ])

            # prediction + probability
            probs = torch.softmax(fusion_logits, dim=1)[:, 1]
            preds = fusion_logits.argmax(dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    # metrics
    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec  = recall_score(all_labels, all_preds, zero_division=0)
    f1   = f1_score(all_labels, all_preds, zero_division=0)
    auc  = roc_auc_score(all_labels, all_probs)
    cm   = confusion_matrix(all_labels, all_preds)

    print("\n===== FINAL FUSION MODEL METRICS =====")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"AUC      : {auc:.4f}")
    print("Confusion Matrix:\n", cm)

    return acc, prec, rec, f1, auc, cm


In [42]:
def evaluate_fusion(loader):
    fusion_head.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Evaluating"):
            imgs = imgs.to(device)
            labels = labels.to(device)

            logits_swin = swin(imgs)
            logits_conv = convnext(imgs)
            logits_vit  = vit(imgs)
            logits_eff  = effnet(imgs)

            fusion_logits = fusion_head([logits_swin, logits_conv, logits_vit, logits_eff])
            probs = F.softmax(fusion_logits, dim=1)[:, 1]  # prob of class 1 (Illicit/Tampered)
            preds = fusion_logits.argmax(dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)

    print("\n=== Fusion Model Evaluation ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"AUC      : {auc:.4f}")
    print("Confusion Matrix:\n", cm)

    return acc, prec, rec, f1, auc, cm


In [47]:
evaluate_fusion(test_loader)


Evaluating Fusion Model: 100%|██████████| 57/57 [01:25<00:00,  1.49s/it]


===== FINAL FUSION MODEL METRICS =====
Accuracy : 0.8303
Precision: 0.5155
Recall   : 0.4236
F1 Score : 0.4650
AUC      : 0.8125
Confusion Matrix:
 [[1364  125]
 [ 181  133]]


(0.8302828618968386,
 0.5155038759689923,
 0.42356687898089174,
 0.46503496503496505,
 np.float64(0.8125040103005908),
 array([[1364,  125],
        [ 181,  133]]))

In [43]:
torch.save(fusion_head.state_dict(), FUSION_SAVE_PATH)
print("Fusion head saved to:", FUSION_SAVE_PATH)


Fusion head saved to: /content/drive/MyDrive/morph_project/models/fusion_4models_logits_head.pth


In [48]:
print("Starting evaluation…")

acc, prec, rec, f1, auc, cm = evaluate_fusion(test_loader)

print("\n=== RESULTS ===")
print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1 Score :", f1)
print("AUC      :", auc)
print("Confusion Matrix:\n", cm)


Starting evaluation…


Evaluating Fusion Model: 100%|██████████| 57/57 [01:13<00:00,  1.29s/it]


===== FINAL FUSION MODEL METRICS =====
Accuracy : 0.8303
Precision: 0.5155
Recall   : 0.4236
F1 Score : 0.4650
AUC      : 0.8125
Confusion Matrix:
 [[1364  125]
 [ 181  133]]

=== RESULTS ===
Accuracy : 0.8302828618968386
Precision: 0.5155038759689923
Recall   : 0.42356687898089174
F1 Score : 0.46503496503496505
AUC      : 0.8125040103005908
Confusion Matrix:
 [[1364  125]
 [ 181  133]]


In [54]:
def train_fusion(num_epochs=5):
    for epoch in range(1, num_epochs + 1):
        fusion_head.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs}"):
            imgs = imgs.to(device)
            labels = labels.to(device)

            # Get logits from frozen models
            with torch.no_grad():
                logits_swin = swin(imgs)
                logits_conv = convnext(imgs)
                logits_vit  = vit(imgs)
                logits_eff  = effnet(imgs)

            fusion_logits = fusion_head([
                logits_swin, logits_conv, logits_vit, logits_eff
            ])

            loss = criterion(fusion_logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)

            preds = fusion_logits.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        # Epoch metrics
        train_loss = running_loss / total
        train_acc  = correct / total

        # Validation metrics
        val_loss, val_acc = validate_fusion()

        # Save metrics
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        print(f"Epoch {epoch}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")


In [52]:
# ----------------------------------------------
# STORE METRICS
# ----------------------------------------------
train_losses = []
train_accs   = []
val_losses   = []
val_accs     = []

# ----------------------------------------------
# VALIDATION FUNCTION used inside training
# ----------------------------------------------
def validate_fusion():
    fusion_head.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            # logits from 4 base models
            logits_swin = swin(imgs)
            logits_conv = convnext(imgs)
            logits_vit  = vit(imgs)
            logits_eff  = effnet(imgs)

            fusion_logits = fusion_head([
                logits_swin, logits_conv, logits_vit, logits_eff
            ])

            loss = criterion(fusion_logits, labels)
            running_loss += loss.item() * imgs.size(0)

            preds = fusion_logits.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss = running_loss / total
    val_acc  = correct / total
    return val_loss, val_acc
